In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'yt-dlp>=2024.11.18',
    'huggingface-hub>=0.26.0',
    'python-dotenv>=1.0.0',
    'pyyaml>=6.0',
    'requests>=2.32.0',
    'soundfile>=0.12.1',
    'numpy>=1.26.0',
], check=True)
subprocess.run(['apt-get', 'install', '-qq', '-y', 'ffmpeg'], check=True)

In [ ]:
import os
import json
import time
import shutil
import threading
import subprocess
import sys
from pathlib import Path
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

import yaml
import requests
import soundfile as sf
import numpy as np
from huggingface_hub import HfApi

WORK_DIR       = Path('/kaggle/working')
DOWNLOAD_DIR   = WORK_DIR / 'raw_downloads'
STANDARD_DIR   = WORK_DIR / 'standardized'
CHECKPOINT_PATH = WORK_DIR / 'checkpoint_p1b.json'
FAILED_LOG     = WORK_DIR / 'failed_downloads.txt'
CONFIG_DIR     = Path('/kaggle/input/datasets/mirza176528/s2s-pipline-v2-0-2/config')

DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
STANDARD_DIR.mkdir(parents=True, exist_ok=True)

TARGET_SR      = 24000
DOWNLOAD_WORKERS = 4
MAX_RETRIES    = 5
SAVE_EVERY     = 20
MIN_DUR_SEC    = 120
MAX_DUR_SEC    = 10800
MIN_FILE_BYTES = 50_000

In [ ]:
def load_secrets():
    try:
        from kaggle_secrets import UserSecretsClient
        c = UserSecretsClient()
        secrets = {
            'HF_TOKEN_PRIMARY':   c.get_secret('HF_TOKEN_PRIMARY'),
            'HF_TOKEN_SECONDARY': c.get_secret('HF_TOKEN_SECONDARY'),
            'HF_TOKEN_TERTIARY':  c.get_secret('HF_TOKEN_TERTIARY'),
            'GEMINI_API_KEY':  c.get_secret('GEMINI_API_KEY'),
        }
        print('[secrets] loaded from Kaggle Secrets')
        return secrets
    except Exception:
        pass

    env_file = Path('.env')
    if env_file.exists():
        from dotenv import load_dotenv
        load_dotenv(env_file)
        print('[secrets] loaded from .env')

    required = ['HF_TOKEN_PRIMARY', 'HF_TOKEN_SECONDARY', 'HF_TOKEN_TERTIARY', 'GEMINI_API_KEY']
    missing = [k for k in required if not os.environ.get(k)]
    if missing:
        raise RuntimeError(f'Missing secrets: {missing}')
    return {k: os.environ[k] for k in required}

SECRETS   = load_secrets()
HF_TOKEN  = SECRETS['HF_TOKEN_PRIMARY']

with open(CONFIG_DIR / 'hf_repos.yaml') as f:
    repos_cfg = yaml.safe_load(f)

STAGE0_REPO = repos_cfg['repos']['stage0_codec']['repo_id']
HF_API      = HfApi(token=HF_TOKEN)
print(f'[config] stage0 repo: {STAGE0_REPO}')

In [ ]:
def load_checkpoint():
    if CHECKPOINT_PATH.exists():
        try:
            with open(CHECKPOINT_PATH) as f:
                state = json.load(f)
            print(f'[checkpoint] local — done={len(state["done"])} failed={len(state["failed"])} standardized={len(state["standardized"])}')
            return state
        except Exception:
            pass

    try:
        url = f'https://huggingface.co/datasets/{STAGE0_REPO}/resolve/main/checkpoint_p1b.json'
        r = requests.get(url, headers={'Authorization': f'Bearer {HF_TOKEN}'}, timeout=30)
        if r.status_code == 200:
            state = r.json()
            with open(CHECKPOINT_PATH, 'w') as f:
                json.dump(state, f)
            print(f'[checkpoint] HF fallback — done={len(state["done"])}')
            return state
    except Exception:
        pass

    print('[checkpoint] fresh start')
    return {
        'done': [],
        'failed': [],
        'standardized': [],
        'stats': {
            'downloaded': 0,
            'standardized': 0,
            'failed_download': 0,
            'failed_standardize': 0,
            'too_short': 0,
            'too_small': 0,
        },
        'last_updated': None,
    }


cp_lock = threading.Lock()

def save_checkpoint(state, upload=False):
    with cp_lock:
        state['last_updated'] = datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ')
        tmp = str(CHECKPOINT_PATH) + '.tmp'
        with open(tmp, 'w') as f:
            json.dump(state, f)
        os.replace(tmp, str(CHECKPOINT_PATH))

    if not upload:
        return

    for attempt in range(6):
        try:
            HF_API.upload_file(
                path_or_fileobj=json.dumps(state).encode(),
                path_in_repo='checkpoint_p1b.json',
                repo_id=STAGE0_REPO,
                repo_type='dataset',
                commit_message='p1b checkpoint',
            )
            return
        except Exception as e:
            wait = min(2 ** attempt, 60)
            print(f'[checkpoint] upload failed attempt {attempt+1}: {e} — retry in {wait}s')
            time.sleep(wait)


state = load_checkpoint()
done_set         = set(state['done'])
standardized_set = set(state['standardized'])

In [ ]:
manifest_local = WORK_DIR / 'video_manifest.jsonl'

if not manifest_local.exists():
    print('[manifest] downloading from HF...')
    for attempt in range(6):
        try:
            url = f'https://huggingface.co/datasets/{STAGE0_REPO}/resolve/main/video_manifest.jsonl'
            r = requests.get(url, headers={'Authorization': f'Bearer {HF_TOKEN}'}, timeout=120, stream=True)
            r.raise_for_status()
            with open(manifest_local, 'wb') as f:
                for chunk in r.iter_content(chunk_size=65536):
                    f.write(chunk)
            print(f'[manifest] downloaded to {manifest_local}')
            break
        except Exception as e:
            wait = min(2 ** attempt, 60)
            print(f'[manifest] download attempt {attempt+1} failed: {e} — retry in {wait}s')
            time.sleep(wait)

all_videos = []
with open(manifest_local, encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            all_videos.append(json.loads(line))

pending = [v for v in all_videos if v['video_id'] not in done_set]

print(f'[manifest] total={len(all_videos)} already_done={len(done_set)} pending={len(pending)}')

In [ ]:
print_lock = threading.Lock()

def tprint(*args):
    with print_lock:
        print(*args, flush=True)


def download_video(video):
    vid_id   = video['video_id']
    url      = f'https://www.youtube.com/watch?v={vid_id}'
    out_path = DOWNLOAD_DIR / f'{vid_id}.wav'

    if out_path.exists() and out_path.stat().st_size > MIN_FILE_BYTES:
        return vid_id, 'already_exists', out_path

    cmd = [
        'yt-dlp',
        '--no-playlist',
        '--no-warnings',
        '--quiet',
        '--format', 'bestaudio/best',
        '--extract-audio',
        '--audio-format', 'wav',
        '--audio-quality', '0',
        '--postprocessor-args', 'ffmpeg:-ar 24000 -ac 1',
        '--output', str(DOWNLOAD_DIR / f'{vid_id}.%(ext)s'),
        '--no-part',
        '--retries', '3',
        '--fragment-retries', '3',
        url,
    ]

    for attempt in range(MAX_RETRIES):
        try:
            result = subprocess.run(cmd, capture_output=True, text=True, timeout=600)

            if result.returncode != 0:
                err = result.stderr.strip()[-200:]
                if 'Video unavailable' in err or 'Private video' in err or 'removed' in err.lower():
                    return vid_id, 'unavailable', None
                if attempt == MAX_RETRIES - 1:
                    return vid_id, f'failed:{err}', None
                time.sleep(5 * (attempt + 1))
                continue

            if not out_path.exists():
                wav_candidates = list(DOWNLOAD_DIR.glob(f'{vid_id}.*'))
                if wav_candidates:
                    wav_candidates[0].rename(out_path)
                else:
                    return vid_id, 'no_output_file', None

            if out_path.stat().st_size < MIN_FILE_BYTES:
                out_path.unlink(missing_ok=True)
                return vid_id, 'too_small', None

            return vid_id, 'ok', out_path

        except subprocess.TimeoutExpired:
            if attempt == MAX_RETRIES - 1:
                return vid_id, 'timeout', None
            time.sleep(10)
        except Exception as e:
            if attempt == MAX_RETRIES - 1:
                return vid_id, f'exception:{e}', None
            time.sleep(5 * (attempt + 1))

    return vid_id, 'max_retries', None


def standardize_audio(vid_id, input_path):
    out_path = STANDARD_DIR / f'{vid_id}.wav'

    if out_path.exists() and out_path.stat().st_size > MIN_FILE_BYTES:
        return vid_id, 'already_exists', out_path

    try:
        with sf.SoundFile(str(input_path)) as f:
            duration = len(f) / f.samplerate

        if duration < MIN_DUR_SEC:
            return vid_id, 'too_short', None

        result = subprocess.run(
            [
                'ffmpeg', '-y',
                '-i', str(input_path),
                '-ar', str(TARGET_SR),
                '-ac', '1',
                '-sample_fmt', 's16',
                '-vn',
                str(out_path),
            ],
            capture_output=True,
            timeout=600,
        )

        if result.returncode != 0:
            return vid_id, f'ffmpeg_error:{result.stderr.decode()[-150:]}', None

        if out_path.stat().st_size < MIN_FILE_BYTES:
            out_path.unlink(missing_ok=True)
            return vid_id, 'output_too_small', None

        input_path.unlink(missing_ok=True)
        return vid_id, 'ok', out_path

    except Exception as e:
        return vid_id, f'standardize_exception:{e}', None


def process_video(video):
    vid_id = video['video_id']

    vid_id, dl_status, dl_path = download_video(video)

    if dl_status not in ('ok', 'already_exists'):
        return vid_id, dl_status, None, 'download'

    if vid_id in standardized_set:
        std_path = STANDARD_DIR / f'{vid_id}.wav'
        if std_path.exists():
            return vid_id, 'already_standardized', std_path, 'standardize'

    vid_id, std_status, std_path = standardize_audio(vid_id, dl_path)
    return vid_id, std_status, std_path, 'standardize'

In [ ]:
total     = len(pending)
completed = 0
success   = 0

print(f'[download] starting {total} videos with {DOWNLOAD_WORKERS} workers\n')

with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as executor:
    futures = {executor.submit(process_video, v): v for v in pending}

    for future in as_completed(futures):
        vid_id, status, out_path, stage = future.result()
        completed += 1

        with cp_lock:
            if status in ('ok', 'already_exists', 'already_standardized'):
                success += 1
                if vid_id not in done_set:
                    done_set.add(vid_id)
                    state['done'].append(vid_id)
                if stage == 'standardize' and vid_id not in standardized_set:
                    standardized_set.add(vid_id)
                    state['standardized'].append(vid_id)
                    state['stats']['standardized'] += 1
                state['stats']['downloaded'] += 1
            else:
                if vid_id not in state['failed']:
                    state['failed'].append(vid_id)
                if stage == 'download':
                    state['stats']['failed_download'] += 1
                    if status == 'too_small':
                        state['stats']['too_small'] += 1
                elif stage == 'standardize':
                    state['stats']['failed_standardize'] += 1
                    if status == 'too_short':
                        state['stats']['too_short'] += 1

        if completed % SAVE_EVERY == 0 or completed == total:
            upload_now = completed % (SAVE_EVERY * 10) == 0
            save_checkpoint(state, upload=upload_now)
            tprint(f'  [{completed}/{total}] ok={success} failed={len(state["failed"])} | '
                   f'dl={state["stats"]["downloaded"]} std={state["stats"]["standardized"]}')

        if status not in ('ok', 'already_exists', 'already_standardized'):
            tprint(f'  [skip] {vid_id} — {status}')

In [ ]:
print('\n[summary]')
print(f'  total processed  : {completed}')
print(f'  standardized ok  : {state["stats"]["standardized"]}')
print(f'  failed download  : {state["stats"]["failed_download"]}')
print(f'  failed standardize: {state["stats"]["failed_standardize"]}')
print(f'  too short        : {state["stats"]["too_short"]}')
print(f'  too small        : {state["stats"]["too_small"]}')

if state['failed']:
    with open(FAILED_LOG, 'w') as f:
        for vid_id in state['failed']:
            f.write(vid_id + '\n')
    print(f'  failed log       : {FAILED_LOG} ({len(state["failed"])} entries)')

ready_files = list(STANDARD_DIR.glob('*.wav'))
total_gb = sum(f.stat().st_size for f in ready_files) / 1024**3
print(f'\n[output] {len(ready_files)} WAV files ready in {STANDARD_DIR}')
print(f'[output] total size: {total_gb:.2f} GB')

save_checkpoint(state, upload=True)
print('\n[done] checkpoint saved — ready for p1c_clean_cpu.ipynb')